<a href="https://colab.research.google.com/github/cicimeilia54-gif/proyek_akuisisi_data/blob/main/akomodasi_kintamani_2501010032.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Langkah 2 — Akuisisi dengan Python (hari 3). Pakai fungsi retry dari Modul 06. Overpass
bisa membalas 429 (terlalu banyak request) atau 504 (server sibuk).


In [18]:
import json, time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import requests

URL = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": "INSTIKI-BIFP243-Praktikum/1.0 (nim@instiki.ac.id)"}
QUERY = """
[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"](area.kec);
out center tags;
"""

def ambil_overpass(query, maks_percobaan=4, jeda=5):
    for percobaan in range(1, maks_percobaan + 1):
        try:
            r = requests.post(URL, data={"data": query},
                              headers=HEADERS, timeout=120)
            if r.status_code == 200:
                return r.json()
            print(f"[percobaan {percobaan}] status {r.status_code}, tunggu {jeda}s")
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[percobaan {percobaan}] {type(e).__name__}, tunggu {jeda}s")
        time.sleep(jeda)
        jeda *= 2
    raise RuntimeError("Gagal mengambil data dari Overpass")

data = ambil_overpass(QUERY)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
Path("data/raw").mkdir(parents=True, exist_ok=True)
raw_path = Path(f"data/raw/osm_akomodasi_kintamani_{stamp}.json")
raw_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print("Jumlah elemen:", len(data["elements"]))

[percobaan 1] status 502, tunggu 5s
[percobaan 2] status 502, tunggu 10s
Jumlah elemen: 32


Langkah 3 — Meratakan menjadi tabel (hari 4). Fungsi ini sudah diuji dengan contoh
respons berstruktur Overpass. Nomor telepon dan email sengaja tidak diambil (minimisasi
data).

In [19]:
def ratakan(elements):
    baris = []
    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)      # node: lat/lon langsung; way: di center
        baris.append({
            "osm_type": e["type"], "osm_id": e["id"],
            "nama": tg.get("name"), "jenis": tg.get("tourism"),
            "lat": pusat.get("lat"), "lon": pusat.get("lon"),
            "desa": tg.get("addr:village"),
            "bintang": tg.get("stars"), "jumlah_kamar": tg.get("rooms"),
            "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e['type']}/{e['id']}",
        })
    return pd.DataFrame(baris)

akomodasi = ratakan(data["elements"])
print(akomodasi["jenis"].value_counts())
print("Tanpa nama:", akomodasi["nama"].isna().sum())

jenis
hotel          16
guest_house     9
hostel          4
chalet          2
motel           1
Name: count, dtype: int64
Tanpa nama: 1


Langkah 4 — Pemeriksaan dasar (hari 4). Periksa minimal: duplikat berdasarkan
osm_type +
osm_id , akomodasi tanpa nama, koordinat di luar wilayah Kintamani, dan
kolom yang hampir selalu kosong (misalnya
diam; catat setiap keputusan di laporan.

In [20]:
# Pemeriksaan Kualitas Data Dasar (Langkah 4)
print("=== CEK DUPLIKAT ===")
jumlah_ganda = akomodasi.duplicated(subset=["osm_type", "osm_id"]).sum()
print("Total baris duplikat:", jumlah_ganda)

print("\n=== CEK NAMA KOSONG ===")
data_tanpa_nama = akomodasi[akomodasi["nama"].isna()].shape[0]
print("Jumlah akomodasi tanpa nama:", data_tanpa_nama)

print("\n=== CEK BATAS KOORDINAT ===")
print("Latitude min/max:", akomodasi["lat"].min(), "s.d.", akomodasi["lat"].max())
print("Longitude min/max:", akomodasi["lon"].min(), "s.d.", akomodasi["lon"].max())

print("\n=== CEK SEL KOSONG PER KOLOM ===")
print(akomodasi.isnull().sum())

=== CEK DUPLIKAT ===
Total baris duplikat: 0

=== CEK NAMA KOSONG ===
Jumlah akomodasi tanpa nama: 1

=== CEK BATAS KOORDINAT ===
Latitude min/max: -8.2837958 s.d. -8.2189435
Longitude min/max: 115.2545822 s.d. 115.4129261

=== CEK SEL KOSONG PER KOLOM ===
osm_type         0
osm_id           0
nama             1
jenis            0
lat              0
lon              0
desa            32
bintang         31
jumlah_kamar    30
website         30
url_osm          0
dtype: int64


6. Struktur dataset target
Dataset akhir data/interim/akomodasi_kintamani.csv memuat minimal kolom berikut.
Data dictionary di docs/data_dictionary.csv wajib menjelaskan setiap kolom dengan
format yang sama seperti tabel ini.

In [21]:
import os

# Membuat folder data/interim jika belum ada
os.makedirs("data/interim", exist_ok=True)

# Menyimpan dataframe ke CSV
akomodasi.to_csv("data/interim/akomodasi_kintamani.csv", index=False, encoding="utf-8")
print("File berhasil disimpan ke data/interim/akomodasi_kintamani.csv")

File berhasil disimpan ke data/interim/akomodasi_kintamani.csv


Langkah 5 - Verifikasi 10 Sampel Akomodasi

In [22]:
# Mengambil 10 sampel acak untuk verifikasi manual
sampel_10 = akomodasi.sample(n=10, random_state=42)

# Menampilkan kolom penting untuk dicek
sampel_10[["nama", "jenis", "lat", "lon", "url_osm"]]

,nama,jenis,lat,lon,url_osm
29,Batur Sunrise Guesthouse,hotel,-8.249020,115.400863,https://www.openstreetmap.org/node/13068182153
15,Caldera,hotel,-8.279139,115.367364,https://www.openstreetmap.org/node/6451134585
24,Bunbulan Panorama Villas,guest_house,-8.222736,115.397971,https://www.openstreetmap.org/node/9769685089
17,Segara Hotel,hotel,-8.278086,115.379134,https://www.openstreetmap.org/node/7058054537
8,The Volcania Guesthouse,hotel,-8.283796,115.365242,https://www.openstreetmap.org/node/5065920543
9,Entrance to: Bunbulan Hill Hostel,hostel,-8.222014,115.395349,https://www.openstreetmap.org/node/5175562421
30,Lava Bali and Hot Spring,hotel,-8.250311,115.396374,https://www.openstreetmap.org/node/13284479326
25,Villa Jempana,hotel,-8.231158,115.410556,https://www.openstreetmap.org/node/9778735917
12,Hut,chalet,-8.242693,115.387587,https://www.openstreetmap.org/node/5479746121
0,The Lakeview Hotel Bali,hotel,-8.283140,115.365844,https://www.openstreetmap.org/node/1977779907


Uji Kelengkapan BPS dan Penyimpanan File Perbandingan

In [23]:
import pandas as pd
import os

# Membuat folder docs jika belum ada
os.makedirs("docs", exist_ok=True)

# Data perbandingan kelengkapan dengan BPS Kintamani
data_perbandingan = {
    "Keterangan": ["Jumlah Data OSM", "Jumlah Data BPS", "Persentase Kelengkapan"],
    "Nilai": [32, 46, "69.57%"]
}

df_bandingan = pd.DataFrame(data_perbandingan)

# Menyimpan ke folder docs
df_bandingan.to_csv("docs/perbandingan_bps.csv", index=False)
print("File perbandingan BPS berhasil disimpan ke docs/perbandingan_bps.csv")

File perbandingan BPS berhasil disimpan ke docs/perbandingan_bps.csv


Langkah 1 — Menentukan wilayah dan jenis akomodasi (hari 2)

In [24]:
import os

# 1. Pastikan folder docs ada
os.makedirs("docs", exist_ok=True)

# 2. Teks query Overpass yang digunakan
query_text = """[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"](area.kec);
out center tags;"""

# 3. Simpan ke dalam file docs/query_overpass.txt
with open("docs/query_overpass.txt", "w", encoding="utf-8") as f:
    f.write(query_text)

print("File docs/query_overpass.txt berhasil dibuat!")

File docs/query_overpass.txt berhasil dibuat!


Langkah 1: Membuat File di Folder docs/

In [25]:
import os
import json

# Pastikan folder docs ada
os.makedirs("docs", exist_ok=True)

# 1. Membuat file docs/provenance.json
provenance_data = {
    "endpoint": "https://overpass-api.de/api/interpreter",
    "query_file": "docs/query_overpass.txt",
    "waktu_akuisisi_utc": "2026-11-05", # Sesuaikan tanggal pengambilan Anda
    "jumlah_elemen": 32,
    "lisensi": "Open Database License (ODbL) - OpenStreetMap contributors",
    "pengambil_data": "Mahasiswa Informatika INSTIKI"
}
with open("docs/provenance.json", "w", encoding="utf-8") as f:
    json.dump(provenance_data, f, ensure_ascii=False, indent=2)

# 2. Membuat file docs/data_dictionary.csv
data_dict_content = """kolom,tipe,deskripsi,contoh,boleh_kosong
osm_type,teks,Jenis objek OSM: node, way, atau relation,way,Tidak
osm_id,bilangan_bulat,ID objek di OpenStreetMap,123456789,Tidak
nama,teks,Nama akomodasi sesuai tag name,Batur Sunrise Guesthouse,Ya
jenis,teks,Nilai tag tourism,hotel,Tidak
lat,lon,desimal,Koordinat titik atau titik tengah bangunan (WGS84),-8.24,115.40,Tidak
desa,teks,Desa sesuai tag addr:village,Batur,Ya
bintang,teks,Klasifikasi bintang (tag stars),3,Ya
jumlah_kamar,teks,Jumlah kamar (tag rooms),20,Ya
website,teks,Situs resmi akomodasi,https://...,Ya
url_osm,teks,Tautan ke objek di openstreetmap.org,https://www.openstreetmap.org/node/...,Tidak
"""
with open("docs/data_dictionary.csv", "w", encoding="utf-8") as f:
    f.write(data_dict_content)

# 3. Membuat file docs/data_ethics_checklist.csv
ethics_content = """aspek_etika,status,keterangan
Batas_wajaran_request,Ya,Menggunakan jeda dan tidak melakukan loop berulang ke server Overpass.
Minimisasi_data_pribadi,Ya,Nomor telepon, email, dan data kontak pribadi pemilik tidak diambil atau disimpan.
Sumber_yang_diizinkan,Ya,Mengambil data murni dari OpenStreetMap dan BPS, menghindari OTA komersial.
Atribusi_lisensi,Ya,Mencantumkan atribusi OpenStreetMap contributors pada laporan dan README.
"""
with open("docs/data_ethics_checklist.csv", "w", encoding="utf-8") as f:
    f.write(ethics_content)

print("Semua file di folder docs/ berhasil dibuat!")

Semua file di folder docs/ berhasil dibuat!


Langkah 2: Membuat File README.md di Root Folder

In [26]:
readme_content = """# Akuisisi Data Akomodasi Kecamatan Kintamani

Repositori ini dibuat untuk memenuhi tugas praktikum mata kuliah **BIFP-243 Akuisisi dan Manajemen Data**.

## Tujuan Tugas
Mengakuisisi data akomodasi (hotel, villa, guest house, dll.) di Kecamatan Kintamani dari OpenStreetMap (OSM) secara etis, membersihkannya, melakukan uji kelengkapan terhadap data resmi BPS, serta mendokumentasikannya secara lengkap[cite: 9, 10, 15].

## Sumber Data
1. **OpenStreetMap (Overpass API):** Data spasial titik akomodasi[cite: 10].
2. **BPS Kabupaten Bangli / Kecamatan Kintamani Dalam Angka:** Sebagai pembanding jumlah akomodasi (46 unit, tingkat kelengkapan 69.57%)[cite: 3, 10].

## Atribusi
> "© OpenStreetMap contributors"[cite: 10, 15]
Lisensi data mengikuti Open Database License (ODbL)[cite: 10, 15].

## Struktur Direktori
- `data/raw/`: Berisi respons mentah JSON dari Overpass API.
- `data/interim/`: Berisi dataset CSV bersih (`akomodasi_kintamani.csv`).
- `notebooks/`: Berisi script Jupyter Notebook utama (`akuisisi_akomodasi.ipynb`).
- `docs/`: Berisi query, provenance, data dictionary, etika data, dan perbandingan BPS.
- `reports/`: Berisi laporan akhir format PDF.

## Cara Menjalankan Ulang
1. Buka file utama di folder `notebooks/` menggunakan Google Colab.
2. Pilih menu **Runtime > Run All** untuk menjalankan seluruh sel dari awal sampai akhir[cite: 16].
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("File README.md berhasil dibuat!")

File README.md berhasil dibuat!


In [27]:
import os
os.makedirs("reports", exist_ok=True)
print("Folder reports berhasil dibuat!")

Folder reports berhasil dibuat!
